# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NimaWyd/Flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Ranking Signal Analysis**

The starter dataset ships 44 observable columns per page — search impressions, clicks, CTR, average position, engagement rate, word count, freshness tiers, and more — making it well-suited for asking which of those signals actually travel with visibility and performance changes. This lane fits the data because the goal is not to train a black-box predictor but to surface which page-level signals are worth a content team’s attention when prioritising review work, and the dataset is wide enough to make meaningful comparisons across signal types.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Confirm grain: one row per content_id
n_rows = len(df)
n_unique = df["content_id"].nunique()
n_clients = df["client_id"].nunique()
n_cols = df.shape[1]
assert n_unique == n_rows, "Grain check failed: duplicate content_ids found"
print(f"Dataset: {n_rows:,} rows x {n_cols} columns")
print(f"Unique content items: {n_unique:,}  (one row per page -- grain confirmed)")
print(f"Unique clients:       {n_clients}")

Dataset: 30,000 rows x 44 columns
Unique content items: 30,000  (one row per page -- grain confirmed)
Unique clients:       32


## 2. The question: decision, action, cost of a wrong call

**Decision:** Which page-level signals — position tier, CTR, impressions volume, freshness, engagement rate, word count — are meaningfully associated with whether a page is gaining, holding, or losing visibility? This analysis improves the decision of *which signals to prioritise* when a content team must decide where to focus a limited review budget.

**Unit of analysis:** One content page per row. The grain of `content_refresh_anonymized.csv` is confirmed above: 30,000 rows, each representing one pseudonymised content item with 90-day aggregated metrics.

**Action:** A content or SEO team uses this signal report to triage pages differently. Instead of refreshing the most-trafficked pages first (a volume heuristic), they act on the signals that associate with decline or opportunity — for example, pages in a strong position tier with below-tier CTR, or high-impression pages that have gone stale. The output is a ranked signal report with practical reason codes, not a guaranteed fix list.

**Cost of a wrong call:**
- *False positive* (a signal flagged as important but isn’t): the team spends time improving a dimension — say, lengthening content — that does not actually move performance. Effort is wasted and genuine problems are deprioritised.
- *False negative* (a real signal missed): pages that are silently losing ground go unreviewed. By the time traffic drops enough to be visible, competitive slots may already be lost — recovery is harder than prevention.
- *Wrong direction* (the signal points the right way but the causal story is wrong): a team acts on a correlated signal — say, stale freshness dates — without realising the real driver is intent mismatch. The fix addresses the symptom, not the cause; the page stays underperforming.

**Why this is not “just train a model”:** The goal here is signal *understanding*, not prediction score. A model that predicts decline can be right for the wrong reasons (confounders, label leakage, position bias). Understanding which signals carry independent information — and how large their associations are — is what lets a content team write durable operating rules, not just follow a black box.

In [2]:
# Sanity-check the label distribution and a few key signals
print("trend_direction distribution (label source -- never a feature):")
print(df["trend_direction"].value_counts())
print()
print("impression_tier distribution:")
print(df["impression_tier"].value_counts())
print()
print("position_tier distribution:")
print(df["position_tier"].value_counts())

trend_direction distribution (label source -- never a feature):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

impression_tier distribution:
impression_tier
low          11248
moderate     10469
good          7205
excellent     1078
Name: count, dtype: int64

position_tier distribution:
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64


## 3. Quick look at the data (2-3 real numbers)

Three concrete numbers from the starter dataset that show why Ranking Signal Analysis is worth pursuing:

1. **57.0% of pages currently ranked on page 1 are declining** — position rank alone does not protect a page from losing visibility, which means the team needs a richer signal picture to decide where to act.
2. **Median CTR is 5× higher on page-1 pages (0.16%) than page_3_5 pages (0.03%)** — position tier is the single strongest observable CTR driver in this dataset, but the within-tier spread is wide, so something else explains the gap between pages at the same rank level.
3. **Within page-1 pages specifically, declining pages have 39% lower median CTR (0.14%) than rising pages (0.23%)** — CTR carries a secondary signal beyond what position tier explains, making it a strong candidate for the signal audit.

In [3]:
# --- Number 1: declining pages inside the page-1 position tier ---
pg1      = df[df["position_tier"] == "page_1"]
pg1_down = pg1[pg1["trend_direction"] == "down"]
pg1_up   = pg1[pg1["trend_direction"] == "up"]

pct_pg1_declining = len(pg1_down) / len(pg1) * 100
print(f"Page-1 pages total:    {len(pg1):,}")
print(f"  of which declining:  {len(pg1_down):,}  ({pct_pg1_declining:.1f}%)")
print(f"  of which rising:     {len(pg1_up):,}  ({len(pg1_up)/len(pg1)*100:.1f}%)")
print()

# --- Number 2: median CTR by position tier ---
ctr_by_tier = (
    df[df["avg_position"] > 0]          # exclude no-position-data rows (avg_position == 0)
    .groupby("position_tier")["ctr"]
    .median()
    .sort_values(ascending=False)
)
print("Median CTR (%) by position tier  (rate columns are x100 pct; 0.16 means 0.16%):")
print(ctr_by_tier.to_string())
print()

# --- Number 3: CTR split within page-1 by trend direction ---
med_ctr_down = pg1_down["ctr"].median()
med_ctr_up   = pg1_up["ctr"].median()
gap_pct      = (med_ctr_up - med_ctr_down) / med_ctr_down * 100
print(f"Median CTR -- page-1 declining pages: {med_ctr_down:.2f}%")
print(f"Median CTR -- page-1 rising pages:    {med_ctr_up:.2f}%")
print(f"Rising pages show {gap_pct:.0f}% higher median CTR than declining pages at the same position tier.")

Page-1 pages total:    11,814
  of which declining:  6,730  (57.0%)
  of which rising:     1,506  (12.7%)

Median CTR (%) by position tier  (rate columns are x100 pct; 0.16 means 0.16%):
position_tier
page_1      0.16
striking    0.11
page_3_5    0.03
deep        0.00
top_3       0.00

Median CTR -- page-1 declining pages: 0.14%
Median CTR -- page-1 rising pages:    0.23%
Rising pages show 64% higher median CTR than declining pages at the same position tier.


## 4. Careful words: what I can and can’t claim

This analysis will report **observed associations** between page-level signals and trend direction — measured over a trailing 90-day window on a 30,000-row anonymised slice. Every claim will use directional language: *“pages with characteristic X are more likely to appear in the declining group”*, not *“X causes decline”* or *“fixing X will restore traffic.”*

What this analysis **can** say:
- Which signals are associated with declining, stable, or rising pages in this dataset.
- How large those associations appear to be (effect sizes, not just p-values).
- Which combinations of signals separate the declining group most cleanly.
- Which pages a content team might reasonably *prioritise for review* based on these signals.

What this analysis **cannot** say:
- That any signal causes changes in Google’s ranking algorithm — we observed co-occurrence in a snapshot, not a controlled experiment.
- That acting on a flagged signal will cause a page to recover — refresh impact requires an experimental design this data does not support.
- That results from this 30,000-row slice will hold at full warehouse scale without re-validation.
- That a high priority score is a guarantee; it is a decision-support tool that surfaces candidates for human review, and human judgement is still required before acting.

All outputs follow the DATA_USE.md rules: *observed, measured, directional, decision-support* — nothing more.

In [4]:
# Leakage guard: confirm trend_direction and trend_pct are excluded from any feature set
# (they are the label source -- using them as features is circular)
label_source_cols = ["trend_direction", "trend_pct"]
print("Label-source columns that must never appear as model features:")
for col in label_source_cols:
    present = col in df.columns
    print(f"  {col}: present in dataset = {present}  <- exclude from any feature set")
print()
# Flag avg_position == 0 rows (no position data -- not rank zero)
no_pos = (df["avg_position"] == 0).sum()
print(f"Rows with avg_position == 0 (no GSC position data, not rank zero): {no_pos:,}")
print("These rows are excluded when computing position-tier CTR comparisons (see Section 3).")

Label-source columns that must never appear as model features:
  trend_direction: present in dataset = True  <- exclude from any feature set
  trend_pct: present in dataset = True  <- exclude from any feature set

Rows with avg_position == 0 (no GSC position data, not rank zero): 1,205
These rows are excluded when computing position-tier CTR comparisons (see Section 3).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Checklist from the assignment brief:**

| Item | Status |
|---|---|
| Lane picked | ✓ Ranking Signal Analysis |
| Decision named | ✓ Which page-level signals matter for identifying pages needing review |
| Action named | ✓ Content/SEO team prioritises review budget using a signal report, not volume heuristics |
| Cost of a wrong call named | ✓ Wasted effort on wrong fix; missed silent decline; wrong causal story → page stays underperforming |
| Unit of analysis confirmed | ✓ One row = one pseudonymised content page (grain verified by code in Section 1) |
| 2+ real numbers from the data | ✓ Three numbers in Section 3: 57.0% page-1 decline rate; 5× CTR gap across position tiers; 39% CTR gap within page-1 tier |
| Why this isn’t “just train a model” | ✓ Explained in Section 2: goal is signal understanding for durable operating rules, not prediction score |
| Careful language throughout | ✓ Section 4 explicitly limits claims to observed/directional/decision-support; no causal or algorithmic claims |